In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: default.

In [2]:
import re

def parse_atom_list(atom_list):
    """
    Parse atom specifications like:
      '1 2 3 5..10 20'
      '[1,2,3,5..10,20]'
      ['1..3']
      ['1..3', 7, '10']
    """

    atoms = []

    # Normalize input to a list of tokens
    if isinstance(atom_list, (list, tuple)):
        tokens = []
        for item in atom_list:
            tokens.extend(re.split(r"[,\s;]+", str(item)))
    else:
        s = atom_list.strip().strip("[]")
        tokens = re.split(r"[,\s;]+", s)

    # Parse tokens
    for tok in tokens:
        if not tok:
            continue
        if ".." in tok:
            a, b = tok.split("..")
            atoms.extend(range(int(a), int(b) + 1))
        else:
            atoms.append(int(tok))

    return sorted(set(atoms))


In [3]:
wc=load_node(81843)

In [4]:
wc.outputs.retrieved.list_object_names()

['_scheduler-stderr.txt',
 '_scheduler-stdout.txt',
 'aiida-1.restart',
 'aiida-pos-1.xyz',
 'aiida.out']

In [5]:
# Get the retrieved folder
retrieved = wc.outputs.retrieved

# Read aiida.out content
with retrieved.open('aiida.out') as handle:
    cp2k_output = handle.read()

# Quick sanity check
#print(aiida_out[:1000])  # print first 1000 characters


In [6]:
def sum_net_charge(cp2k_output, atom_list, section):
    atoms = parse_atom_list(atom_list)
    lines = cp2k_output.splitlines()
    section = section.upper()

    # --- find section header ---
    start = None
    for i, line in enumerate(lines):
        if section == "MULLIKEN" and "MULLIKEN POPULATION ANALYSIS" in line.upper():
            start = i
            break
        if section == "HIRSHFELD" and "HIRSHFELD CHARGES" in line.upper():
            start = i
            break

    if start is None:
        raise ValueError(f"{section} section not found")

    total_charge = 0.0

    # --- parse numeric table lines only ---
    for line in lines[start + 1:]:
        line = line.strip()
        if not line:
            continue

        fields = line.split()

        # table rows ALWAYS start with atom index
        if not fields[0].isdigit():
            # once we have started reading atoms, this means table is over
            if total_charge != 0.0:
                break
            continue

        atom_id = int(fields[0])
        if atom_id not in atoms:
            continue
        if section == "MULLIKEN":
            # Net charge is penultimate column
            net_charge = float(fields[-2])
        else:  # HIRSHFELD
            # Net charge is last column
            net_charge = float(fields[-1])

        total_charge += net_charge

    return total_charge


In [7]:
sum_net_charge(cp2k_output, ['1..93'], 'MULLIKEN')

1.0511439999999999

In [8]:
sum_net_charge(cp2k_output, ['1..93'], 'HIRSHFELD')

6.958